# Baseline Training — All Models
Trains each baseline model sequentially, saves checkpoints, metrics, and figures per model.

In [ ]:
# Cell 1: Imports & Setup
import os, sys, math, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.notebook import tqdm

PROJECT_ROOT = Path(os.path.abspath('')).parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'model'))
print(f'Project root: {PROJECT_ROOT}')
print(f'Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

In [ ]:
# Cell 2: Load Dataset
from src.dataset import AIDetectionDataset, ImageTransform, create_dataloaders
from src.config import Config

cfg = Config()
cfg.training.epochs = 1
cfg.training.image_size = 384
cfg.training.batch_size = 8
cfg.training.mixed_precision = False
cfg.training.num_workers = 0
cfg.training.val_check_interval = 50
cfg.training.gradient_accumulation_steps = 1
cfg.dataset.val_split = 0.1
cfg.dataset.test_split = 0.1

full_dataset = AIDetectionDataset(
    root_dir=str(PROJECT_ROOT),
    metadata_paths=cfg.dataset.metadata_paths,
    transform=None,
    is_train=True,
    size=cfg.training.image_size,
    undersample=True,
)

print(f'Total samples: {len(full_dataset)}')
print(f'Real: {sum(1 for _, l in full_dataset.samples if l==0)}')
print(f'AI:   {sum(1 for _, l in full_dataset.samples if l==1)}')

# --- TEST MODE: 5K subset ---
from sklearn.model_selection import train_test_split
SAMPLE_SIZE = 5000
labels_arr = np.array([s[1] for s in full_dataset.samples])
indices = np.arange(len(full_dataset))
sampled_idx, _ = train_test_split(indices, train_size=min(SAMPLE_SIZE, len(full_dataset)),
    stratify=labels_arr, random_state=42)
print(f"Test mode: {len(sampled_idx)} images (from {len(full_dataset)})")
full_dataset.samples = [full_dataset.samples[i] for i in sampled_idx]


In [ ]:
# Cell 3: Split into Train/Val/Test
from sklearn.model_selection import train_test_split

labels = [s[1] for s in full_dataset.samples]
indices = list(range(len(full_dataset)))

train_idx, temp_idx = train_test_split(
    indices, test_size=cfg.dataset.val_split + cfg.dataset.test_split,
    stratify=labels, random_state=42,
)

temp_labels = [labels[i] for i in temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx, test_size=cfg.dataset.test_split / (cfg.dataset.val_split + cfg.dataset.test_split),
    stratify=temp_labels, random_state=42,
)
print(f'Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}')

train_dataset = AIDetectionDataset(
    root_dir=str(PROJECT_ROOT), metadata_paths=[],
    transform=ImageTransform(size=cfg.training.image_size, augment=True),
    is_train=True, size=cfg.training.image_size, undersample=False,
)
train_dataset.samples = [full_dataset.samples[i] for i in train_idx]

val_dataset = AIDetectionDataset(
    root_dir=str(PROJECT_ROOT), metadata_paths=[],
    transform=ImageTransform(size=cfg.training.image_size, augment=False),
    is_train=False, size=cfg.training.image_size, undersample=False,
)
val_dataset.samples = [full_dataset.samples[i] for i in val_idx]

test_dataset = AIDetectionDataset(
    root_dir=str(PROJECT_ROOT), metadata_paths=[],
    transform=ImageTransform(size=cfg.training.image_size, augment=False),
    is_train=False, size=cfg.training.image_size, undersample=False,
)
test_dataset.samples = [full_dataset.samples[i] for i in test_idx]

train_loader = DataLoader(train_dataset, batch_size=cfg.training.batch_size, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=cfg.training.batch_size, shuffle=False, num_workers=0, pin_memory=False)
test_loader = DataLoader(test_dataset, batch_size=cfg.training.batch_size, shuffle=False, num_workers=0, pin_memory=False)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}')

In [ ]:
# Cell 4: Define All Baselines
from src.baselines import (
    SimpleCNN, LightViT, count_parameters,
    resnet18, resnet50, efficientnet_b0, vit_b_16, swin_t,
    CLIPBaseline, FreqDetect, deit_small,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}\n')

BASELINE_REGISTRY = {
    'SimpleCNN':      lambda: SimpleCNN(),
    'LightViT':       lambda: LightViT(img_size=cfg.training.image_size, depth=4, num_heads=4, embed_dim=192),
    'ResNet-18':      lambda: resnet18(),
    'ResNet-50':      lambda: resnet50(),
    'EfficientNet-B0': lambda: efficientnet_b0(),
    'ViT-B/16':       lambda: vit_b_16(img_size=cfg.training.image_size),
    'Swin-T':         lambda: swin_t(),
    'DeiT-S':         lambda: deit_small(img_size=cfg.training.image_size),
    'CLIP':           lambda: CLIPBaseline(img_size=cfg.training.image_size),
    'FreqDetect':     lambda: FreqDetect(img_size=cfg.training.image_size),
}

x = torch.randn(2, 3, cfg.training.image_size, cfg.training.image_size)
print(f"{'Model':<20} {'Params':>10} {'Output':>10}")
print('-' * 42)
for name, fn in BASELINE_REGISTRY.items():
    m = fn()
    p = count_parameters(m)
    o = list(m(x).shape)
    print(f'{name:<20} {p:>10,}  {str(o):>10}')

# ── Select which baselines to train ──
# Set MODELS_TO_TRAIN = list(BASELINE_REGISTRY.keys()) for all, or pick a subset:
MODELS_TO_TRAIN = list(BASELINE_REGISTRY.keys())
# e.g. MODELS_TO_TRAIN = ['ResNet-50', 'EfficientNet-B0']
print(f'\nWill train: {MODELS_TO_TRAIN}')

In [ ]:
# Cell 5: Training Loop for All Baselines
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, LinearLR, SequentialLR
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.calibration import calibration_curve
from sklearn.metrics import confusion_matrix

NUM_EPOCHS = cfg.training.epochs
criterion = nn.CrossEntropyLoss(label_smoothing=cfg.training.label_smoothing)

for model_name in MODELS_TO_TRAIN:
    print('\n' + '='*70)
    print(f'Training {model_name}...')
    print('='*70)

    model = BASELINE_REGISTRY[model_name]().to(device)
    print(f'Parameters: {count_parameters(model):,}')

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.training.lr, weight_decay=cfg.training.weight_decay)
    warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=min(500, len(train_loader)))
    cosine = CosineAnnealingWarmRestarts(optimizer, T_0=NUM_EPOCHS * len(train_loader), T_mult=2, eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[min(500, len(train_loader))])

    ckpt_dir = PROJECT_ROOT / 'model' / 'checkpoints' / 'test' / 'baselines_model' / f'{model_name.lower().replace("/", "_").replace("-", "_")}'
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_acc = 0
    best_epoch = -1

    for epoch in range(NUM_EPOCHS):
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        pbar = tqdm(train_loader, desc=f'{model_name} Epoch {epoch+1}/{NUM_EPOCHS}')
        for images, labels in pbar:
            try:
            except Exception as e:
                print(f"  Warning: skipping bad batch: {e}")
                optimizer.zero_grad()
                continue
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            scheduler.step()

            total_loss += loss.item()
            preds = logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            pbar.set_postfix({'loss': f'{total_loss/(total/cfg.training.batch_size):.4f}',
                             'acc': f'{correct/total*100:.2f}%',
                             'lr': f'{scheduler.get_last_lr()[0]:.2e}'})

        train_acc = correct / total * 100
        history["train_acc"].append(train_acc)
        history["train_loss"].append(total_loss / len(train_loader))

        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                try:
                except Exception as e:
                    print(f"  Warning: bad val batch: {e}")
                    continue
                images, labels = images.to(device), labels.to(device)
                logits = model(images)
                loss = criterion(logits, labels)
                val_loss += loss.item()
                preds = logits.argmax(dim=-1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total * 100
        history["val_acc"].append(val_acc)
        history["val_loss"].append(val_loss / len(val_loader))
        print(f'  train={train_acc:.2f}%, val={val_acc:.2f}%')

        if val_acc > best_acc:
            best_acc = val_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), ckpt_dir / 'best.pt')
            print(f'  * Saved best ({best_acc:.2f}%)')

    print(f'\n{model_name} done. Best val acc: {best_acc:.2f}% at epoch {best_epoch}')

    # Save final model
    torch.save(model.state_dict(), ckpt_dir / 'final.pt')

    # Save history
    results_dir = PROJECT_ROOT / 'paper' / 'result' / 'test' / 'baselines_model' / model_name.lower().replace('/', '_').replace('-', '_')
    results_dir.mkdir(parents=True, exist_ok=True)
    with open(results_dir / 'history.json', 'w') as f:
        json.dump(history, f, indent=2)

    # Evaluate on test set
    model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            try:
            except Exception as e:
                print(f"  Warning: bad test batch: {e}")
                continue
            images = images.to(device)
            logits = model(images)
            probs = F.softmax(logits, dim=-1)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    y_true = np.array(all_labels)
    y_score = np.array(all_probs)
    y_pred = (y_score >= 0.5).astype(int)

    acc = (y_pred == y_true).mean() * 100
    prec = precision_score(y_true, y_pred, zero_division=0) * 100
    rec = recall_score(y_true, y_pred, zero_division=0) * 100
    f1 = f1_score(y_true, y_pred, zero_division=0) * 100
    auc = roc_auc_score(y_true, y_score)

    metrics = {
        'model': model_name,
        'params': count_parameters(model),
        'best_val_acc': round(best_acc, 2),
        'best_epoch': best_epoch,
        'test_accuracy': round(acc, 2),
        'test_precision': round(prec, 2),
        'test_recall': round(rec, 2),
        'test_f1': round(f1, 2),
        'test_auc': round(auc, 4),
    }
    with open(results_dir / 'metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2)

    print(f'Test: acc={acc:.2f}%, prec={prec:.2f}%, rec={rec:.2f}%, f1={f1:.2f}%, auc={auc:.4f}')
    print(f'Results saved to {results_dir}/')

In [ ]:
# Cell 6: Summary Table of All Baselines
all_metrics = []
for model_name in MODELS_TO_TRAIN:
    results_dir = PROJECT_ROOT / 'paper' / 'result' / 'test' / 'baselines_model' / model_name.lower().replace('/', '_').replace('-', '_')
    metrics_file = results_dir / 'metrics.json'
    if metrics_file.exists():
        with open(metrics_file) as f:
            all_metrics.append(json.load(f))

if all_metrics:
    df = pd.DataFrame(all_metrics).set_index('model')
    print('\n=== BASELINE COMPARISON ===')
    print(df.to_string())
    summary_path = PROJECT_ROOT / 'paper' / 'result' / 'test' / 'baselines_model' / 'baseline_summary.csv'
    df.to_csv(summary_path)
    print(f'\nSummary saved to {summary_path}')